# Experiment 1: Inspect the Sparse Representation

In [67]:
import gzip
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from scipy.sparse import save_npz
file_path= '/Users/cuongtran/Downloads/c4-train.00000-of-01024-30K.json.gz'
corpus = []
with gzip.open(file_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        if 'text' in data:
            corpus.append(data['text'].strip())
print(f"Đã load xong dữ liệu! Tổng số documents: {len(corpus)}")

Đã load xong dữ liệu! Tổng số documents: 30000


In [68]:
c_vectorizer = CountVectorizer()
TF = c_vectorizer.fit_transform(corpus)
tfidf_transformer = TfidfTransformer()
TF_IDF_matrix = tfidf_transformer.fit_transform(TF)
IDF = tfidf_transformer.idf_
save_npz("../tf_idf_matrix_c4.npz", TF_IDF_matrix)
joblib.dump(c_vectorizer, "../tfidf_vectorizer.joblib")


['tfidf_vectorizer.joblib']

In [69]:
N = TF_IDF_matrix.shape[0]
V = TF_IDF_matrix.shape[1]
print(f"Number of documents (N) = {N}")
print(f"Vocabulary size (V) = {V}")
print(f"Matrix shape = {TF_IDF_matrix.shape}")


Number of documents (N) = 30000
Vocabulary size (V) = 193540
Matrix shape = (30000, 193540)


In [70]:
nnz_X = TF_IDF_matrix.nnz
S = 1.0 - (nnz_X / (N * V))
print(f"Số phần tử khác 0 (nnz(X)) : {nnz_X}")
print(f"Độ thưa Sparsity (S)       : {S:.6f} (hoặc {S * 100:.4f}%)")

Số phần tử khác 0 (nnz(X)) : 4985822
Độ thưa Sparsity (S)       : 0.999141 (hoặc 99.9141%)


Dù một văn bản chỉ chứa rất ít từ, vector đại diện của nó bắt buộc phải giữ nguyên chiều (V) vì 3 lý do cốt lõi:

- Chuẩn hóa không gian đặc trưng: Mỗi vị trí trong vector tương ứng cố định với một từ trong toàn bộ từ điển. Việc cùng chiều (V) giúp đồng bộ dữ liệu giữa tất cả các văn bản.

- Yêu cầu của thuật toán: Các vector bắt buộc phải có cùng số chiều thì mới thực hiện được các phép toán so sánh (như tính độ tương đồng Cosine) hoặc làm đầu vào cho mô hình Machine Learning.

- Không gây tốn tài nguyên: Hệ thống lưu trữ dưới dạng ma trận thưa (sparse matrix), tức là chỉ ghi nhớ những từ thực sự có xuất hiện (giá trị khác 0), bỏ qua phần lớn các số 0 nên không lãng phí bộ nhớ.

In [71]:
feature_names = np.array(c_vectorizer.get_feature_names_out())

In [72]:
df_array = (TF > 0).sum(axis=0).A1
top20_df_idx = np.argsort(df_array)[::-1][:20]
df_top20 = pd.DataFrame({
    'Term (Top DF)': feature_names[top20_df_idx],
    'Document Frequency': df_array[top20_df_idx]
})
print("1. TOP 20 TERMS THEO DOCUMENT FREQUENCY (DF)")
display(df_top20)

1. TOP 20 TERMS THEO DOCUMENT FREQUENCY (DF)


,Term (Top DF),Document Frequency
0,the,27893
1,and,27423
2,to,26689
3,of,26031
4,in,25224
5,for,23651
6,is,22739
7,with,21405
8,on,20262
9,that,18370


In [73]:
top20_idf_idx = np.argsort(IDF)[::-1][:20]
idf_top20 = pd.DataFrame({
    'Term (Top IDF)': feature_names[top20_idf_idx],
    'IDF Score': IDF[top20_idf_idx]
})
print("2. TOP 20 TERMS CÓ IDF CAO NHẤT")
display(idf_top20)

2. TOP 20 TERMS CÓ IDF CAO NHẤT


,Term (Top IDF),IDF Score
0,00000,10.615839
1,00003,10.615839
2,000040,10.615839
3,00005,10.615839
4,0000856166,10.615839
5,0001042,10.615839
6,000116,10.615839
7,00012,10.615839
8,00015,10.615839
9,00016,10.615839


In [74]:
doc_idx = 0
doc_tfidf_array = TF_IDF_matrix[doc_idx].toarray().flatten()
top20_tfidf_idx = np.argsort(doc_tfidf_array)[::-1][:20]
tfidf_top20 = pd.DataFrame({
    'Term (Top TF-IDF in Doc)': feature_names[top20_tfidf_idx],
    'TF-IDF Score': doc_tfidf_array[top20_tfidf_idx]
})
print(f"3. TOP 20 TERMS CÓ TF-IDF CAO NHẤT TRONG DOCUMENT #{doc_idx}")
display(tfidf_top20)

3. TOP 20 TERMS CÓ TF-IDF CAO NHẤT TRONG DOCUMENT #0


,Term (Top TF-IDF in Doc),TF-IDF Score
0,bbq,0.461808
1,class,0.275842
2,meat,0.194158
3,balay,0.179403
4,kcbs,0.172551
5,lonestar,0.167689
6,will,0.154676
7,missoula,0.150594
8,apron,0.141357
9,smoker,0.140491


- Một term xuất hiện nhiều trong corpus chưa chắc có TF-IDF cao, chẳng hạn có thể tồn tại term xuất hiện gần như mọi document trong corpus, nhờ đó mà tăng giá trị DF của term lên, nhưng đồng thời giá trị IDF giảm đi, từ đó giá trị TF-IDF của term trong document có thể thấp và ngược lại, có thể có term chỉ xuất hiện trong rất ít document, dẫn đến việc kéo giá trị IDF đi lên, nhờ đó mà giá trị TF-IDF của term trong document sẽ cao hơn.
- Một term có IDF cao chưa chắc có giá trị TF-IDF cao trong mọi document. Chẳng hạn sẽ có term xuất hiện nhiều ở document này, dẫn đến TF của term trong document sẽ cao hơn, dẫn đến giá trị TF-IDF của term sẽ cao, nhưng xét trên document khác, cũng là term đang xét nhưng tần suất xuất hiện ít hơn, từ đó mà TF của nó sẽ thấp hơn, và giá trị TF-IDF sẽ giảm đi.